In [2]:
import numpy as np

In [3]:
import numpy as np

rng = np.random.default_rng(7)
bikes = rng.poisson(10, size=(5, 24))

In [4]:
## RETO 8

salidas  = np.array([5, 8, 3, 6, 2, 4])
llegadas = np.array([2, 4, 9, 3, 7, 5]) 

inicial   = 12
capacidad = 20

neto = llegadas - salidas

saldo = inicial + np.cumsum(neto)
saldo = np.clip(saldo, 0, capacidad)

print(saldo)

[ 9  5 11  8 13 14]


In [5]:
## RETO 9

media_estacion = bikes.mean(axis=1)
hora_max = bikes.argmax(axis=1)

In [6]:
import time

M = np.random.default_rng(0).random((2000, 500))

# Con bucles
t0 = time.perf_counter()
medias_bucle = []
for fila in M:
    total = 0.0
    for x in fila:
        total += x
    medias_bucle.append(total / len(fila))
t_bucle = time.perf_counter() - t0

# Con NumPy
t0 = time.perf_counter()
medias_np = M.mean(axis=1)
t_np = time.perf_counter() - t0

print(np.allclose(medias_bucle, medias_np))   # comprobar que dan lo mismo
print(f"Bucle: {t_bucle:.3f} s | NumPy: {t_np:.5f} s | {t_bucle/t_np:.0f} veces más rápido")

True
Bucle: 0.104 s | NumPy: 0.00059 s | 176 veces más rápido


In [7]:
import pandas as pd

In [ ]:
stations = pd.read_csv("Archivos/stations.csv")

In [17]:
# stations.shape
# stations.head()
# stations.tail()
# stations.info()
# stations.sample()
# stations.describe()
# stations.isna()
# stations.duplicated()
# stations.nunique()
# stations.value_counts()

In [18]:
## RETO 1

import numpy as np
import pandas as pd

mini = pd.DataFrame({
    "station": ["Xàtiva", "Colón", "Ruzafa"],
    "capacity": [20, 15, 25],
    "bikes": [12, 3, 18],
})

mini["occupancy"] = mini["bikes"] / mini["capacity"]

mini["status"] = np.where(mini["occupancy"] < 0.3, "low", "ok")
mini

,station,capacity,bikes,occupancy,status
0,Xàtiva,20,12,0.60,ok
1,Colón,15,3,0.20,low
2,Ruzafa,25,18,0.72,ok


In [23]:
## RETO 2

stations = pd.read_csv("Archivos/stations.csv")
usage = pd.read_csv("Archivos/usage_hourly.csv", parse_dates=["timestamp"])
weather = pd.read_json("Archivos/weather_daily.json")

formas = [stations.shape, usage.shape, weather.shape]
formas

[(60, 7), (43350, 5), (30, 6)]

In [24]:
## RETO 3

print("Duplicados en usage:", usage.duplicated().sum())
print("Nulos en bicis disponibles:", usage["bikes_available"].isna().sum())
print("Nulos en capacidad:", stations["capacity"].isna().sum())
print("Distritos distintos:", stations["district"].nunique())

Duplicados en usage: 150
Nulos en bicis disponibles: 870
Nulos en capacidad: 2
Distritos distintos: 19


In [25]:
## RETO 4

n_999 = (usage["rentals"] == 999).sum()
n_neg = (usage["returns"] < 0).sum()

print("Filas con 999 alquileres:", n_999)
print("Filas con devoluciones negativas:", n_neg)

Filas con 999 alquileres: 12
Filas con devoluciones negativas: 8


In [28]:
## RETO 5

mask = (stations["district"] == "Ciutat Vella") & (stations["capacity"] >= 30)

ids = sorted(stations.loc[mask, "station_id"].tolist())
ids

[2, 3, 6, 8]

In [ ]:
## RETO 6

validas = usage[(usage["rentals"] != 999) & (usage["returns"] >= 0)]

top = validas.sort_values(
    by=["returns", "timestamp", "station_id"],
    ascending=[False, True, True],
)

top.head(1)

,station_id,timestamp,rentals,returns,bikes_available
18016,60,2026-09-02 08:00:00,6,29,20.0


In [ ]:
## RETO 7

clean = usage.copy()
n0 = len(clean)

clean = clean.drop_duplicates()
n1 = len(clean)

clean = clean[
    clean["rentals"].between(0, 100) & clean["returns"].between(0, 100)
]
n2 = len(clean)

print("Duplicados quitados:", n0 - n1)
print("Imposibles quitados:", n1 - n2)
print("Total quitado:", n0 - n2, "| quedan", n2)

Duplicados quitados: 150
Imposibles quitados: 20
Total quitado: 170 | quedan 43180


In [ ]:
## RETO 8

clean["timestamp"] = pd.to_datetime(clean["timestamp"])

clean["date"] = clean["timestamp"].dt.date
clean["hour"] = clean["timestamp"].dt.hour
clean["weekday"] = clean["timestamp"].dt.day_name()
clean["is_weekend"] = clean["timestamp"].dt.dayofweek >= 5

In [ ]:
## RETO 9

clean = clean.sort_values(["station_id", "timestamp"])

clean["bikes_available"] = (
    clean.groupby("station_id")["bikes_available"].ffill()
)
clean["bikes_available"] = (
    clean.groupby("station_id")["bikes_available"].bfill()
)

clean["bikes_available"].isna().sum()   # debería dar 0

np.int64(0)

In [ ]:
## RETO 10

stations["district"] = stations["district"].str.strip().str.title()


print(stations["district"].nunique(), "\n",
stations["district"].value_counts())

stations["capacity"] = stations["capacity"].fillna(
    stations.groupby("district")["capacity"].transform("median")
)

print(stations["capacity"].isna().sum()) 

13 
 district
Ciutat Vella        8
Eixample            6
Poblats Marítims    6
Extramurs           5
La Saïdia           5
El Pla Del Real     5
Quatre Carreres     5
Campanar            4
Camins Al Grau      4
L'Olivereta         3
Patraix             3
Jesús               3
Algirós             3
Name: count, dtype: int64
0
